In [29]:
#ライブラリをimport
import os
from datetime import datetime
from glob import glob

from dotenv import load_dotenv
import json
import pandas as pd
from sqlalchemy import create_engine

In [30]:
#データベースへのパスを作成
load_dotenv()
data_base = os.getenv("DATABASE_URL")
engine = create_engine(f"{data_base}")

In [31]:
#保存したデータからcategory_idと対応するtitleを取得
path = "../data/raw/category_data.json"
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)
items = data.get("items", [])
category_dic={}
for item in items:
    category_dic[item.get("id")] = [item.get("snippet", {}).get("title")]

In [32]:
#保存した人気動画のデータから各種情報を抽出
videos_info_list=[]
files = glob("../data/raw/popular_videos/*.json")
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
    items=data.get("items", [])
    for item, i in zip(items, range(1,51)):
        snippet = item.get("snippet", {})
        statistics = item.get("statistics", {})
        category_id = snippet.get("categoryId")
        video_info={
            "daytime":datetime.strptime(file[42:57], "%Y%m%d_%H%M%S"),
            "video_id" : item.get("id"),
            "title" : snippet.get("title"),
            "channel_id": snippet.get("channelId"),
            "channel_title": snippet.get("channelTitle"),
            "category_title":category_dic[f"{category_id}"],
            "view_count" : statistics.get("viewCount"),
            "like_count" : statistics.get("likeCount"),
            "favorite_count" : statistics.get("favoriteCount"),
            "comment_count" : statistics.get("commentCount")
        }
        videos_info_list.append(video_info)
df=pd.DataFrame(videos_info_list)
#SQL形式でデータベースに保存
df.to_sql("popular_videos", engine, if_exists="replace", index=False)
#CSV形式で保存
df.to_csv("../data/processed/popular_videos.csv", encoding="utf-8-sig",index=False)

In [33]:
#保存した人気動画のチャンネルデータから各種情報を抽出
channel_info_list=[]
channel_files = glob("../data/raw/channels_statistics/*.json")
for channel_file in channel_files:
    with open(channel_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        snippet = item.get("snippet", {})
        statistics = item.get("statistics", {})
        channel_info = {
            "daytime":datetime.strptime(channel_file[51:66], "%Y%m%d_%H%M%S"),
            "channel_id" : item.get("id"),
            "channel_title" : snippet.get("title"),
            "channel_view_count" : statistics.get("viewCount"),
            "subscriber_count" : statistics.get("subscriberCount"),
            "video_count":statistics.get("videoCount")
        }
        channel_info_list.append(channel_info)
df2 = pd.DataFrame(channel_info_list)
#SQL形式でデータベースに保存
df2.to_sql('popular_channels', engine, if_exists='replace', index=False)
#CSV形式で保存
df2.to_csv("../data/processed/popular_channels.csv", encoding="utf-8-sig",index=False)

In [34]:
#保存したすべての動画のデータから各種情報を抽出
video_info_list=[]
video_files = glob("../data/raw/videos_statistics/*.json")
for video_file in video_files:
    with open(video_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        statistics = item.get("statistics", {})
        video_info = {
            "daytime":datetime.strptime(video_file[47:62], "%Y%m%d_%H%M%S"),
            "video_id" : item.get("id"),
            "view_count" : statistics.get("viewCount"),
            "like_count" : statistics.get("likeCount"),
            "favorite_count":statistics.get("favoriteCount"),
            "comment_count":statistics.get("commentCount")
        }
        video_info_list.append(video_info)
df3 = pd.DataFrame(video_info_list)
#SQL形式でデータベースに保存
df3.to_sql('video_statistics', engine, if_exists='replace', index=False)
#CSV形式で保存
df3.to_csv("../data/processed/video_statistics.csv", encoding="utf-8-sig",index=False)